In [24]:
import math
#binomial option pricing for european options
def european_call_price(S0,K,r,sigma,T,N):
    dt= T/N
    u= math.exp(sigma*math.sqrt(dt))
    d= 1/u
    p = (math.exp(r * dt) - d) / (u - d)

    #maturity stock prices
    stock_prices = []
    for j in range(N + 1):

        stock_price = S0 * (u ** j) * (d ** (N - j))

        stock_prices.append(stock_price)

    #compute payoffs max(S-K,0)
    payoffs= []
    for S in stock_prices:
        payoff= max(S-K,0)
        payoffs.append(payoff)
    

    #backward induction
    while len(payoffs) > 1:
        new_values = []

        for i in range(len(payoffs) - 1):

            value = math.exp(-r * dt) * (
                p * payoffs[i + 1]
                + (1 - p) * payoffs[i]
            )

            new_values.append(value)

        payoffs = new_values
    return payoffs[0]
    


In [41]:
#binomial option pricing for non dividend paying american call options
def american_call_price(S0,K,r,sigma,T,N):
    dt= T/N
    u= math.exp(sigma*math.sqrt(dt))
    d= 1/u
    p = (math.exp(r * dt) - d) / (u - d)
    
    #building stock tree
    stock_tree=[]
    for i in range(N+1):
        #maturity stock prices
        stock_prices = []
        for j in range(i + 1):

            stock_price = S0 * (u ** j) * (d ** (i - j))

            stock_prices.append(stock_price)
        stock_tree.append(stock_prices)
    

    #compute payoffs
    payoffs= []
    for S in stock_tree[N]:
        payoff= max(S-K,0)
        payoffs.append(payoff)
    
    
    for i in range(N-1, -1, -1):

        new_values = []

        for j in range(i + 1):

            continuation = math.exp(-r * dt) * (
                p * payoffs[j + 1]
                + (1 - p) * payoffs[j]
            )

            exercise = max(stock_tree[i][j] - K, 0)
            if exercise > continuation:
                print(i, j, stock_tree[i][j], exercise, continuation)
           

            option_value = continuation

            new_values.append(option_value)

        payoffs = new_values

    return payoffs[0]
    




   

    

    

In [42]:
europrice= european_call_price(100,120,0.05,0.2,1,100)
americanprice = american_call_price(100,120,0.05,0.2,1,100)
print("european call price=", europrice, "american call price=", americanprice)


european call price= 3.255832704534697 american call price= 3.255832704534697


In [29]:
import math
#binomial option pricing for european put
def european_put_price(S0,K,r,sigma,T,N):
    dt= T/N
    u= math.exp(sigma*math.sqrt(dt))
    d= 1/u
    p = (math.exp(r * dt) - d) / (u - d)

    #maturity stock prices
    stock_prices = []
    for j in range(N + 1):

        stock_price = S0 * (u ** j) * (d ** (N - j))

        stock_prices.append(stock_price)

    #compute payoffs max(S-K,0)
    payoffs= []
    for S in stock_prices:
        payoff= max(K-S,0)
        payoffs.append(payoff)
    

    #backward induction
    while len(payoffs) > 1:
        new_values = []

        for i in range(len(payoffs) - 1):

            value = math.exp(-r * dt) * (
                p * payoffs[i + 1]
                + (1 - p) * payoffs[i]
            )

            new_values.append(value)

        payoffs = new_values
    return payoffs[0]
    


In [45]:
#binomial option pricing for non dividend paying american put options
def american_put_price(S0,K,r,sigma,T,N):
    dt= T/N
    u= math.exp(sigma*math.sqrt(dt))
    d= 1/u
    p = (math.exp(r * dt) - d) / (u - d)
    
    #building stock tree
    stock_tree=[]
    for i in range(N+1):
        #maturity stock prices
        stock_prices = []
        for j in range(i + 1):

            stock_price = S0 * (u ** j) * (d ** (i - j))

            stock_prices.append(stock_price)
        stock_tree.append(stock_prices)
    

    #compute payoffs
    payoffs= []
    for S in stock_tree[N]:
        payoff= max(K-S,0)
        payoffs.append(payoff)
    
    
    for i in range(N - 1, -1, -1):

        new_values = []

        for j in range(i + 1):

            continuation = math.exp(-r * dt) * (
                p * payoffs[j + 1]
                + (1 - p) * payoffs[j]
            )

            exercise = max(K- stock_tree[i][j], 0)
            

            option_value = max(exercise, continuation)

            new_values.append(option_value)

        payoffs = new_values

    return payoffs[0]
    


In [46]:
europutprice= european_put_price(100,95,0.10,0.2,1,100)
americanputprice = american_put_price(100,95,0.10,0.2,1,100)
print("european put price=", europutprice, "american put price=", americanputprice)


european put price= 2.403663698298644 american put price= 2.9773976595271288


In [6]:
#European options pricing using finite differences method
import numpy as np
import math

def euro_fd_call(S0, K, r, sigma, T, M, N):
        Smax = 2 * K
        dS = Smax / M
        dt = T / N
        stock_prices= np.linspace(0,Smax,M+1)
        V= np.zeros((M+1,N+1))

        for i in range(M+1):
            V[i,N] = max(stock_prices[i] - K,0)

        V[0, :] = 0

        #backward induction
        for n in range(N+1):
            tau = T - n * dt
            V[M, n] = Smax - K * math.exp(-r * tau)

        for n in range(N - 1, -1, -1):

            for i in range(1, M):

                a = 0.5 * dt * (sigma**2 * i**2 - r * i)
                b = 1 - dt * (sigma**2 * i**2 + r)
                c = 0.5 * dt * (sigma**2 * i**2 + r * i)

                V[i, n] = (
                    a * V[i - 1, n + 1]
                    + b * V[i, n + 1]
                    + c * V[i + 1, n + 1]
                )

        return np.interp(S0, stock_prices, V[:, 0]) 
        


In [9]:
#american options using finite differences method
import numpy as np
import math

def american_fd_call(S0, K, r, sigma, T, M, N):
        Smax = 2 * K
        dS = Smax / M
        dt = T / N
        stock_prices= np.linspace(0,Smax,M+1)
        V= np.zeros((M+1,N+1))

        for i in range(M+1):
            V[i,N] = max(stock_prices[i] - K,0)

        V[0, :] = 0

        #backward induction
        for n in range(N+1):
            tau = T - n * dt
            V[M, n] = Smax - K * math.exp(-r * tau)

        for n in range(N - 1, -1, -1):
        
                    for i in range(1, M):
        
                        a = 0.5 * dt * (sigma**2 * i**2 - r * i)
                        b = 1 - dt * (sigma**2 * i**2 + r)
                        c = 0.5 * dt * (sigma**2 * i**2 + r * i)
        
                        continuation = (
                            a * V[i - 1, n + 1]
                            + b * V[i, n + 1]
                            + c * V[i + 1, n + 1]
                        )
                        exercise= max(stock_prices[i]-K,0)
                        V[i, n] = max(exercise, continuation)
        return np.interp(S0, stock_prices, V[:, 0])

In [10]:
eurofinite = euro_fd_call(100,120,0.05,0.2, 1, 10, 100)
americanfinite = american_fd_call(100,120,0.05,0.2, 1, 10, 100)
print("European call price:", eurofinite,"american call price:", americanfinite)

European call price: 3.5345295726126804 american call price: 3.5345295726126804
